# L18 · frozen layers and repeated few-shot SetFit

Follow data → objective → gradients → checkpoint → held-out predictions → reusable artifact. No API or paid provider. CPU training is intentionally small, with original or licensed data and visible failures.


## Set up

Run `uv sync --locked --project examples/training/representation` and select that Python 3.11 kernel. Start in this project or its notebooks directory. The pinned tiny BERT weights are about 17.8 MB; actual SetFit additionally uses about 70 MB of pretrained MiniLM sentence weights. First downloads need network; later use HF_HUB_OFFLINE=1. CPU/float32/one thread is the verified profile; GPU runs are not claimed. Read README for measured resources and exact package/data/model pins. Restart and Run All.


In [ ]:
import sys,json
from pathlib import Path
project=Path.cwd()
if project.name=='notebooks':project=project.parent
root=project.parents[2]
sys.path.insert(0,str(project/'src'))
fixture=json.loads((root/'examples/data/fixtures/embedding-training.json').read_text())
from collections import Counter
print('Original embedding splits:',Counter(r['split']for r in fixture['rows']))
print('Original policy:',fixture['policy'])
print('Review data:',json.loads((root/'examples/data/manifests/sentiment.json').read_text())['license'])


## Predict before training

The fixed 200/40/80 review splits are reused from E4. Three seeds each select eight examples per class; every method receives identical selected IDs within a seed. Compare full, partial and head-only BERT, lexical and frozen sentence baselines, and real SetFit.

The next cell imports the actual training implementation. It initializes a fresh model, performs real optimization and validation selection, checks intended gradients/updates and frozen parameters where applicable, then saves/reloads model artifacts.


In [ ]:
from atlas_representation.classification import main
main()

## Inspect real results and errors

Task metrics, loss curves, source IDs, actual predictions and limits are retained. The test set never chooses a checkpoint or tunes a hyperparameter. A poor result is still valid execution evidence.


In [ ]:
artifact_path=root/'examples/artifacts/e7-classification/results.json'
result=json.loads(artifact_path.read_text())
print(json.dumps(result["summary"],indent=2))
print("First freeze report:",result["runs"][0]["parameters"])

## Account for work and reproducibility

Compare optimizer steps, batches, selected checkpoint, model calls, trainable counts, float32 CPU placement and saved artifact hashes. A small fixed-step run is not a broad benchmark or convergence claim. SetFit uses fixed training steps; the BERT/embedding paths select from declared validation checkpoints. Independent review and beginner feedback remain pending.


In [ ]:
assert json.loads(artifact_path.read_text())==result
print('Saved result reload checked; the source also checks actual model reload equivalence.')

## Transfer exercise

Find a seed where adapting the sentence encoder underperforms the frozen encoder. Explain why pair multiplication does not increase the number of independent labels. Distinguish BERT/MiniLM architecture differences from a controlled objective comparison.

Report: data IDs and provenance; baseline; changed component; prediction; actual learning curve; frozen/update checks; selected checkpoint; untouched evaluation; representative failure; runtime; artifact reload; next fresh-data test.

### Solution guidance

Use source IDs and tensor/mask boundaries before interpreting a score. Do not relabel test data as training-only, silver as gold, dependent pairs as new independent labels, a projection as full geometry, or a surrogate loss decrease as task success.


## Verification boundary

All code cells run in a fresh actual kernel. Notebooks and CLI import one implementation. Run notebooks before final CLI --record commands because they overwrite timing-bearing artifacts. Original prose/code are MIT, licensed review data retain attribution, and book pages/artwork are not redistributed.
